# Pokemon TCG AI Battle — Agent Testing

Run this notebook inside a Kaggle competition notebook. The cabt engine is pre-installed there.

**Steps:**
1. Set `GITHUB_URL` to your repository.
2. Edit `AGENTS` to list the agent class names you want to test.
3. Run all cells.

## 1. Clone repo

In [ ]:
GITHUB_URL = "https://github.com/<YOUR_USERNAME>/<YOUR_REPO>.git"  # replace this

!git clone {GITHUB_URL} /kaggle/working/repo

In [ ]:
import os
import sys

os.chdir("/kaggle/working/repo")
sys.path.insert(0, "/kaggle/working/repo/src")

## 2. Available deck

Card IDs in the engine's bundled default deck.

In [ ]:
from collections import Counter
from kaggle_environments.envs.cabt.cabt import deck as DEFAULT_DECK

counts = Counter(DEFAULT_DECK)
print(f"Default deck — {len(DEFAULT_DECK)} cards total\n")
print(f"{'Card ID':<10} {'Count'}")
print("-" * 20)
for card_id, count in sorted(counts.items()):
    print(f"{card_id:<10} {count}")

## 3. Inspect option format

Run this section **once** to see the raw structure of the options the engine
passes to agents. Use the output to fill in `src/ptcg/rules/schema.py`.
You can skip this section once the schema is filled in.

In [ ]:
from kaggle_environments import make
from ptcg.agents.inspector_agent import InspectorAgent

inspector = InspectorAgent()
inspector.on_game_start()

env = make("cabt", configuration={"decks": [inspector.get_deck(), inspector.get_deck()]})
env.run([inspector, inspector])

inspector.on_game_end({})

In [ ]:
# Adjust max_turns to see more or fewer turns
inspector.print_options(max_turns=5)

## 4. Configure agents

Add the class names of the agents you want to test. Each name must match a class
in `src/ptcg/agents/` using the snake_case naming convention:
`RandomAgent` → `src/ptcg/agents/random_agent.py`,
`RuleBasedAgent` → `src/ptcg/agents/rule_based_agent.py`.

In [ ]:
AGENTS = [
    "RandomAgent",
    # "RuleBasedAgent",  # uncomment once schema.py is filled in
]

## 5. Run battles

Every agent in the list plays against every other agent (including itself).
The last battle's environment is stored in `last_env` for log inspection below.

In [ ]:
import importlib
import re
from kaggle_environments import make


def load_agent(class_name: str):
    module_name = re.sub(r"(?<!^)(?=[A-Z])", "_", class_name).lower()
    module = importlib.import_module(f"ptcg.agents.{module_name}")
    return getattr(module, class_name)()


def run_battle(agent0, agent1):
    env = make("cabt", configuration={"decks": [agent0.get_deck(), agent1.get_deck()]})
    env.run([agent0, agent1])
    rewards = [step["reward"] for step in env.steps[-1]]
    return rewards, env


results = []
last_env = None

for name0 in AGENTS:
    for name1 in AGENTS:
        agent0 = load_agent(name0)
        agent1 = load_agent(name1)

        agent0.on_game_start()
        agent1.on_game_start()

        rewards, last_env = run_battle(agent0, agent1)
        result = {"steps": last_env.steps, "rewards": rewards}

        agent0.on_game_end(result)
        agent1.on_game_end(result)

        results.append({"agent0": name0, "agent1": name1, "rewards": rewards})
        print(f"{name0} vs {name1}  →  {name0}: {rewards[0]}  |  {name1}: {rewards[1]}")

## 6. Results summary

In [ ]:
print(f"{'Agent 0':<20} {'Agent 1':<20} {'Reward 0':>10} {'Reward 1':>10}")
print("-" * 65)
for r in results:
    print(f"{r['agent0']:<20} {r['agent1']:<20} {r['rewards'][0]:>10} {r['rewards'][1]:>10}")

## 7. Game logs (last battle)

Two views of what happened in the last battle:
- **Step logs** — the game events reported at each turn.
- **Full history** — every turn's observation and the actions both agents took.

In [ ]:
# Step logs — one block of events per turn
for i, step in enumerate(last_env.steps):
    for player_state in step:
        obs = player_state.observation
        logs = obs.get("logs") or []
        if logs:
            print(f"--- turn {i} ---")
            for entry in logs:
                print(f"  {entry}")
            break  # both players share the same logs; only print once per turn

In [ ]:
import json

# Full history — observation + actions at every turn
history = last_env.steps[0][0].get("visualize")

if history is None:
    print("No visualize data available.")
else:
    for i, turn in enumerate(history):
        action = turn.get("action")
        obs    = turn.get("obs", {})
        logs   = (obs.get("logs") or []) if obs else []
        print(f"=== turn {i}  action={action} ===")
        for entry in logs:
            print(f"  {entry}")